# SupportSphere — Mistral-7B QLoRA Fine-tuning (T4 Verified)

**What was wrong with previous notebooks:**
- Notebook 1: Pinned old `bitsandbytes` + `triton` versions against Colab's modern torch 2.8 → import crash
- Notebook 2: Used `bf16=True` — T4 GPUs do **not** support bfloat16 (only A100/H100 do). PyTorch silently fell back to CPU → 31 hour estimate

**This notebook fixes both issues.**

### Before running:
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Fill in Cell 2 with your HuggingFace credentials
3. Runtime → Run all


In [ ]:
# ── CELL 2: YOUR CREDENTIALS ──────────────────────────────────
HF_TOKEN = "hf_YOUR_TOKEN_HERE"
HF_USERNAME = "ai-support-platform"         # your HuggingFace username
REPO_NAME   = "supportsphere-mistral-support"

HF_REPO_ID  = f"{HF_USERNAME}/{REPO_NAME}"
print(f"Model will be uploaded to: https://huggingface.co/{HF_REPO_ID}")


Model will be uploaded to: https://huggingface.co/ai-support-platform/supportsphere-mistral-support


## Step 1 — Verify GPU and Install Dependencies

**Key rules that fix the previous errors:**
- Do NOT install or pin `triton` — Colab's torch already has the right version bundled
- Do NOT reinstall `torch` — same reason  
- Use `bitsandbytes>=0.43` (CUDA-kernel path, no triton dependency)
- Use `fp16=True` not `bf16=True` — T4 supports fp16, not bf16


In [2]:
# ── CELL 3: VERIFY GPU FIRST ──────────────────────────────────
import torch, platform

print(f"Python : {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.version.cuda}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime > Change runtime type > T4 GPU and restart."
    )

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU    : {gpu}")
print(f"VRAM   : {vram:.1f} GB")

# T4 does NOT support bfloat16 — this check prevents the silent CPU fallback
if not torch.cuda.is_bf16_supported():
    print("bf16 NOT supported on this GPU — will use fp16 (correct for T4)")
else:
    print("bf16 supported (A100/H100)")

# Confirm CUDA is actually usable
x = torch.tensor([1.0]).cuda()
print(f"CUDA test: {x} — GPU is active")


Python : 3.12.13
PyTorch: 2.11.0+cu128
CUDA   : 12.8
GPU    : Tesla T4
VRAM   : 15.6 GB
bf16 supported (A100/H100)
CUDA test: tensor([1.], device='cuda:0') — GPU is active


In [3]:
# ── CELL 4: INSTALL DEPENDENCIES ──────────────────────────────
# Rules:
# 1. Do NOT touch torch or triton — Colab's preinstalled versions match the GPU driver
# 2. Install latest stable bitsandbytes — uses CUDA kernels, no triton dependency
# 3. Pin transformers/peft/trl to versions verified to work together on Colab 2025/2026

!pip install -q -U \
    "bitsandbytes>=0.43.0" \
    "transformers>=4.40.0" \
    "peft>=0.10.0" \
    "trl>=0.8.6" \
    "accelerate>=0.29.0" \
    "datasets>=2.19.0" \
    "huggingface_hub>=0.22.0" \
    "sentencepiece" \
    "scipy"

print("Done installing")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.1 MB/s eta 0:00:00
Done installing


In [4]:
# ── CELL 5: SANITY CHECK IMPORTS ──────────────────────────────
# If this cell fails: Runtime > Restart session, then re-run from Cell 3
# (pip sometimes needs a fresh process to load new binary extensions)
import torch
import bitsandbytes as bnb
import transformers, peft, trl, accelerate

print(f"bitsandbytes : {bnb.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"accelerate   : {accelerate.__version__}")

# This is the critical test — if this imports fine, the triton issue is gone
from transformers import BitsAndBytesConfig
print("BitsAndBytesConfig import OK — no triton conflict")

# Verify bitsandbytes can see the GPU
print(f"bitsandbytes CUDA available: {torch.cuda.is_available()}")


bitsandbytes : 0.49.2
transformers : 5.12.1
peft         : 0.19.1
trl          : 1.7.0
accelerate   : 1.14.0
BitsAndBytesConfig import OK — no triton conflict
bitsandbytes CUDA available: True


In [5]:
# ── CELL 6: LOGIN TO HUGGINGFACE ──────────────────────────────
from huggingface_hub import login, create_repo

login(token=HF_TOKEN)
print("Logged in to HuggingFace")

try:
    create_repo(HF_REPO_ID, private=False, exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{HF_REPO_ID}")
except Exception as e:
    print(f"Repo note: {e}")


Logged in to HuggingFace
Repo note: (Request ID: Root=1-6a3e71bc-571dd78046eeb67a0dbe0e8e;8df320c3-e9b3-48ce-9208-77031d2abf06)

403 Forbidden: You don't have the rights to create a model under the namespace "ai-support-platform".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.


## Step 2 — Load and Format Dataset

In [6]:
# ── CELL 7: LOAD BITEXT DATASET ───────────────────────────────
from datasets import load_dataset
import pandas as pd

print("Downloading Bitext customer support dataset...")
raw = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split="train",
)
print(f"Loaded {len(raw)} examples")
print(f"Columns: {raw.column_names}")
print()

# Show intent distribution
df = raw.to_pandas()
print("Intent distribution:")
print(df["intent"].value_counts().head(15).to_string())


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Loaded 26872 examples
Columns: ['flags', 'instruction', 'category', 'intent', 'response']

Intent distribution:
intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998


In [7]:
# ── CELL 8: FORMAT FOR MISTRAL ────────────────────────────────
# Mistral instruction format: <s>[INST] question [/INST] answer</s>
# The space before </s> is intentional — matches Mistral's training format

def format_for_mistral(example):
    instruction = example["instruction"].strip()
    response    = example["response"].strip()

    # Filter noise: skip very short examples
    if len(instruction) < 15 or len(response) < 20:
        return {"text": ""}

    text = f"<s>[INST] {instruction} [/INST] {response}</s>"
    return {"text": text}

formatted = raw.map(
    format_for_mistral,
    remove_columns=raw.column_names,
)

# Remove empty (filtered) examples
formatted = formatted.filter(lambda x: len(x["text"]) > 50)

# 98% train, 2% eval (eval is just for loss monitoring, keep it small)
split = formatted.train_test_split(test_size=0.02, seed=42)
train_data = split["train"]
eval_data  = split["test"]

print(f"Train: {len(train_data)} examples")
print(f"Eval : {len(eval_data)} examples")
print()
print("Example formatted entry:")
print(train_data[0]["text"][:400])


Map:   0%|          | 0/26872 [00:00<?, ? examples/s]

Filter:   0%|          | 0/26872 [00:00<?, ? examples/s]

Train: 26282 examples
Eval : 537 examples

Example formatted entry:
<s>[INST] purchase {{Order Number}} status [/INST] Appreciate your message to inquire about the status of your purchase with the purchase number {{Order Number}}. I'm here to provide you with the information you need. To check the current status of your purchase, I recommend visiting the "Order Status" or "Order History" section on our website. By navigating to this section, you will be able to tr


## Step 3 — Load Mistral-7B in 4-bit

**Why 4-bit?** Mistral-7B in full fp16 = ~14GB. T4 has 16GB VRAM.
After accounting for gradients and optimizer states, that OOMs.
4-bit NF4 quantization brings the base model to ~4GB, leaving plenty of room for LoRA adapters + activations.


In [8]:
# ── CELL 9: LOAD MODEL IN 4-BIT ───────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "mistralai/Mistral-7B-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,      # ← ADD THIS: forces all non-quantized layers to fp16
    trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Force the model's own dtype config to fp16 (Mistral-7B-v0.3 defaults to bf16)
model.config.torch_dtype = torch.float16

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"VRAM used: {vram_used:.2f} GB")
print(f"Model device: {next(model.parameters()).device}")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/137k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

VRAM used: 4.14 GB
Model device: cuda:0


In [9]:
# ── CELL 10: ATTACH LORA ADAPTERS ─────────────────────────────
from peft import LoraConfig, prepare_model_for_kbit_training

# Prepare for k-bit training (enables gradient checkpointing hooks)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                       # rank — 16 is good balance of quality vs VRAM on T4
    lora_alpha=32,              # scaling = alpha/r = 2 (standard)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[            # all linear layers — paper shows this matches full fine-tune quality
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

# NOTE: We pass lora_config to SFTTrainer below, NOT to get_peft_model here
# Reason: SFTTrainer needs to call prepare_model internally in the right order
# to avoid the "frozen adapter" bug (trl issue #3926)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"After prepare_model_for_kbit_training:")
print(f"  Trainable: {trainable_params:,} ({100*trainable_params/total_params:.3f}%)")
print(f"  (LoRA adapters will be added by SFTTrainer in next step)")


After prepare_model_for_kbit_training:
  Trainable: 0 (0.000%)
  (LoRA adapters will be added by SFTTrainer in next step)


## Step 4 — Train

**Critical T4 settings:**
- `fp16=True` not `bf16=True` — this was the cause of the 31-hour estimate
- `per_device_train_batch_size=2` with `gradient_accumulation_steps=8` → effective batch = 16
- `max_steps=500` → ~2 hours on T4, gives good results on 27k dataset
- `paged_adamw_8bit` → memory-efficient optimizer
- `gradient_checkpointing=True` → trades compute for VRAM (essential on T4)


In [10]:
# ── CELL 11: CONFIGURE TRAINING ───────────────────────────────
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/content/mistral-support-checkpoints"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # ── Duration ───────────────────────────────────────────
    max_steps=500,                  # ~2 hours on T4; full 1 epoch = ~1700 steps

    # ── Batch / Memory ─────────────────────────────────────
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # effective batch = 2*8 = 16
    gradient_checkpointing=True,

    # ── Optimizer ──────────────────────────────────────────
    optim="paged_adamw_8bit",       # 8-bit optimizer saves VRAM
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.001,

    # ── Precision ──────────────────────────────────────────
    fp16=True,                      # T4 supports fp16 — do NOT use bf16 on T4
    # bf16=False,                   # already False by default; shown for clarity

    # ── Sequence length ────────────────────────────────────
    max_length=512,             # keep at 512 for T4 VRAM budget
    dataset_text_field="text",
    packing=False,

    # ── Logging / Saving ───────────────────────────────────
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    # ── Misc ───────────────────────────────────────────────
    report_to="none",               # disable wandb
    seed=42,
)

print("Training config ready")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Max steps: {training_args.max_steps}")
print(f"fp16: {training_args.fp16}  bf16: {training_args.bf16}")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training config ready
Effective batch size: 16
Max steps: 500
fp16: True  bf16: False


In [11]:
# ── CELL 12: CREATE TRAINER AND START TRAINING ────────────────
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/content/mistral-support-checkpoints"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_steps=500,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # ← ADD THIS
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.001,
    fp16=False,                      # T4 uses fp16
    bf16=False,                     # ← explicitly False
    max_length=512,
    dataset_text_field="text",
    packing=False,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=lora_config,
    args=training_args,
)

trainer.model.print_trainable_parameters()
print(f"Model device: {next(trainer.model.parameters()).device}")
print(f"VRAM before training: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print()
print("Starting training...")
print("-" * 60)

train_result = trainer.train()

print(f"Final loss : {train_result.training_loss:.4f}")
print(f"Steps      : {train_result.global_step}")
print(f"Time       : {train_result.metrics['train_runtime']/60:.1f} minutes")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/26282 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/26282 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/26282 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/537 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/537 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754
Model device: cuda:0
VRAM before training: 4.76 GB

Starting training...
------------------------------------------------------------


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.797133,0.824136,0.834123,244832.000000,0.759691
200,0.763956,0.775080,0.755927,489889.000000,0.771901
300,0.714210,0.735336,0.758115,734709.000000,0.782797
400,0.701162,0.710509,0.728865,977702.000000,0.788821
500,0.687715,0.706866,0.732252,1225681.000000,0.790826


Final loss : 0.7742
Steps      : 500
Time       : 189.8 minutes


In [12]:
# ── CELL 13: SAVE LORA ADAPTER ────────────────────────────────
# Save the adapter weights first (small, ~300MB)
# This is a safety checkpoint before the heavier merge step
ADAPTER_DIR = "/content/lora-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved to {ADAPTER_DIR}")

import os
adapter_size_mb = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
) / 1e6
print(f"Adapter size: {adapter_size_mb:.1f} MB")


LoRA adapter saved to /content/lora-adapter
Adapter size: 87.6 MB


In [13]:
# ── CELL 14: QUICK INFERENCE TEST (before merge) ──────────────
test_prompts = [
    "I want to cancel my order, how do I do that?",
    "I was charged twice for the same item.",
    "My app keeps crashing when I try to log in.",
    "Where is my order? It was supposed to arrive yesterday.",
    "I want a refund for my recent purchase.",
]

print("Testing fine-tuned model (LoRA adapter active):")
print("=" * 60)

for prompt in test_prompts:
    formatted = f"<s>[INST] {prompt} [/INST]"
    inputs = tokenizer(formatted, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    print(f"Q: {prompt}")
    print(f"A: {response}")
    print("-" * 60)


Testing fine-tuned model (LoRA adapter active):


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Q: I want to cancel my order, how do I do that?
A: I've picked up that you're looking to cancel your order and I'm here to assist you. To cancel your order, please follow these steps:

1. Sign in to your account: Access our platform by signing in to your {{Online Company Portal Info}}.
2. Navigate to your order history: Once you're logged in, go to the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section.
------------------------------------------------------------
Q: I was charged twice for the same item.
A: I'm sorry to hear that you've been charged twice for the same item. I understand how frustrating this can be. To assist you further, could you please provide me with the details of the item and the dates of the charges? This will help me investigate the issue and ensure that the charges are corrected. Thank you for bringing this to our attention, and I'll do my best to resolve this matter for you.
------------------------------------------------------------
Q: 

## Step 5 — Merge LoRA into Base Model and Upload

Merging requires reloading the base model in fp16 (not 4-bit).
We free the training model from VRAM first to make room.


In [15]:
# ── RUN THIS BEFORE CELL 15 ───────────────────────────────────
!pip install -q -U torchao
import importlib, peft, transformers
importlib.reload(peft)
print("torchao upgraded — now run Cell 15")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.3 MB/s eta 0:00:00
torchao upgraded — now run Cell 15


In [1]:
import os
import gc
import torch
import safetensors.torch as st
from huggingface_hub import snapshot_download
from safetensors import safe_open
from peft import PeftConfig

ADAPTER_DIR = "/content/lora-adapter"
BASE_MODEL = "mistralai/Mistral-7B-v0.3"
MERGED_DIR = "/content/merged-model"
os.makedirs(MERGED_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Why this approach: merge_and_unload() and any device_map="auto" load both
# need the ENTIRE fp16 base model (~14-15GB) resident in memory at once,
# which exceeds both free-tier Colab's ~12-13GB system RAM and the T4's
# 15GB VRAM the moment you add any overhead on top. This script instead
# processes ONE safetensors shard file at a time: load only that shard's
# tensors, apply the LoRA delta to whichever of its weights are LoRA
# targets, write the merged shard straight to disk, then free everything
# before moving to the next shard. Peak memory is bounded by the size of
# a single shard (~2GB) plus the small LoRA adapter (~100-200MB), not the
# full model.
# ---------------------------------------------------------------------------

print("Step 1: downloading base model safetensors shards (no loading into memory yet)...")
base_model_path = snapshot_download(BASE_MODEL, allow_patterns=["*.safetensors", "*.json", "*.model"])
print("Base model files at:", base_model_path)

print("\nStep 2: loading LoRA adapter weights (small, safe to load fully)...")
peft_config = PeftConfig.from_pretrained(ADAPTER_DIR)
lora_alpha = peft_config.lora_alpha
lora_r = peft_config.r
scaling = lora_alpha / lora_r
print(f"LoRA scaling factor (alpha/r): {scaling}")

adapter_weights = st.load_file(os.path.join(ADAPTER_DIR, "adapter_model.safetensors"))
print(f"Loaded {len(adapter_weights)} adapter tensors")
print("Sample adapter keys:", list(adapter_weights.keys())[:4])

# Build a lookup: base weight name -> (lora_A tensor, lora_B tensor)
# Real PEFT adapter keys look like:
#   base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
# (the ".default." segment is the adapter name PEFT assigns by default).
# The corresponding base model weight key (as stored in the base model's
# own safetensors files) is:
#   model.layers.0.self_attn.q_proj.weight
lora_pairs = {}
for key in adapter_weights:
    if ".lora_A." in key:
        # Strip the PEFT wrapper prefix "base_model.model." and the
        # ".lora_A.<adapter_name>.weight" suffix down to ".weight"
        stripped = key.replace("base_model.model.", "", 1)
        # stripped is now: model.layers.0.self_attn.q_proj.lora_A.default.weight
        base_key = stripped.split(".lora_A.")[0] + ".weight"
        b_key = key.replace(".lora_A.", ".lora_B.")
        lora_pairs[base_key] = (adapter_weights[key], adapter_weights[b_key])

print(f"Found {len(lora_pairs)} LoRA-targeted weight matrices to merge")
print("Sample target keys:", list(lora_pairs.keys())[:4])
if len(lora_pairs) == 0:
    raise RuntimeError(
        "No LoRA pairs matched — adapter key format didn't parse as expected. "
        "Print adapter_weights.keys() and check the format before continuing; "
        "merging with zero matches would silently produce an unmodified base model."
    )

# ---------------------------------------------------------------------------
# Step 3: find all safetensors shard files and the index mapping which
# tensor lives in which shard
# ---------------------------------------------------------------------------
index_file = os.path.join(base_model_path, "model.safetensors.index.json")
if os.path.exists(index_file):
    import json
    with open(index_file) as f:
        index = json.load(f)
    shard_files = sorted(set(index["weight_map"].values()))
else:
    # single-shard model
    shard_files = [f for f in os.listdir(base_model_path) if f.endswith(".safetensors")]

print(f"\nStep 3: processing {len(shard_files)} shard(s) one at a time...\n")

for shard_name in shard_files:
    shard_path = os.path.join(base_model_path, shard_name)
    print(f"  Loading shard: {shard_name}")

    merged_tensors = {}
    with safe_open(shard_path, framework="pt", device="cpu") as f:
        for tensor_name in f.keys():
            tensor = f.get_tensor(tensor_name).to(torch.float16)

            if tensor_name in lora_pairs:
                lora_A, lora_B = lora_pairs[tensor_name]
                # delta W = scaling * B @ A, applied as W + delta
                lora_A = lora_A.to(torch.float16)
                lora_B = lora_B.to(torch.float16)
                delta = (lora_B @ lora_A) * scaling
                tensor = tensor + delta
                del delta

            merged_tensors[tensor_name] = tensor.contiguous()

    out_path = os.path.join(MERGED_DIR, shard_name)
    st.save_file(merged_tensors, out_path)
    print(f"  Saved merged shard to {out_path}")

    # Free this shard's memory completely before moving to the next one
    del merged_tensors
    gc.collect()

print("\nStep 4: copying config/tokenizer files...")
import shutil
for fname in os.listdir(base_model_path):
    if fname.endswith((".json", ".model")) and fname != "model.safetensors.index.json":
        shutil.copy(os.path.join(base_model_path, fname), os.path.join(MERGED_DIR, fname))

# Copy the index file too if it exists (needed for multi-shard loading)
if os.path.exists(index_file):
    shutil.copy(index_file, os.path.join(MERGED_DIR, "model.safetensors.index.json"))

print(f"\nDone. Merged model saved to {MERGED_DIR}")
print("Peak memory stayed bounded to roughly one shard + adapter size throughout.")

Step 1: downloading base model safetensors shards (no loading into memory yet)...


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Base model files at: /root/.cache/huggingface/hub/models--mistralai--Mistral-7B-v0.3/snapshots/caa1feb0e54d415e2df31207e5f4e273e33509b1

Step 2: loading LoRA adapter weights (small, safe to load fully)...
LoRA scaling factor (alpha/r): 2.0
Loaded 448 adapter tensors
Sample adapter keys: ['base_model.model.model.layers.0.mlp.down_proj.lora_A.weight', 'base_model.model.model.layers.0.mlp.down_proj.lora_B.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.weight']
Found 224 LoRA-targeted weight matrices to merge
Sample target keys: ['model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.self_attn.k_proj.weight']

Step 3: processing 3 shard(s) one at a time...

  Loading shard: model-00001-of-00003.safetensors
  Saved merged shard to /content/merged-model/model-00001-of-00003.safetensors
  Loading shard: model-00002-of-00003.safetensors
  Sa

In [4]:
from transformers import AutoModelForCausalLM

merged = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [15]:
from huggingface_hub import whoami

print(whoami(token=HF_TOKEN))
HF_REPO_ID = "AyeshaImtiaz/supportsphere-mistral-support"

print(HF_REPO_ID)

{'type': 'user', 'id': '680ba7bfead7436e45442052', 'name': 'AyeshaImtiaz', 'fullname': 'ayesha imtiaz', 'isPro': False, 'avatarUrl': '/avatars/c2aa6548e93601bbb41c7e1ffdfb7ca0.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'ai-support-platform', 'role': 'fineGrained', 'createdAt': '2026-06-25T14:01:46.936Z', 'fineGrained': {'canReadGatedRepos': False, 'global': [], 'scoped': [{'entity': {'_id': '65770c3426ef61bbf101d4da', 'type': 'model', 'name': 'mistralai/Mistral-7B-Instruct-v0.2'}, 'permissions': ['repo.access.read', 'repo.content.read', 'repo.write']}, {'entity': {'_id': '680ba7bfead7436e45442052', 'type': 'user', 'name': 'AyeshaImtiaz'}, 'permissions': ['user.webhooks.read', 'repo.content.read', 'repo.access.read', 'repo.write', 'user.webhooks.write', 'collection.read', 'collection.write']}]}}}}
AyeshaImtiaz/supportsphere-mistral-support


In [17]:
# ── CELL 16: UPLOAD TO HUGGINGFACE HUB ───────────────────────
print(f"Uploading to https://huggingface.co/{HF_REPO_ID}")
print("This uploads ~14GB — takes 15-30 minutes on Colab...")

# merged.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.3")

tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)

print(f"Upload complete!")
print(f"Your model: https://huggingface.co/{HF_REPO_ID}")
print()
print("Add this to your .env file:")
print(f"HF_MODEL_ID={HF_REPO_ID}")


Uploading to https://huggingface.co/AyeshaImtiaz/supportsphere-mistral-support
This uploads ~14GB — takes 15-30 minutes on Colab...


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Upload complete!
Your model: https://huggingface.co/AyeshaImtiaz/supportsphere-mistral-support

Add this to your .env file:
HF_MODEL_ID=AyeshaImtiaz/supportsphere-mistral-support


In [ ]:
# ── CELL 17: WRITE MODEL CARD ─────────────────────────────────
from huggingface_hub import HfApi

card = f"""---
language:
- en
- ur
license: apache-2.0
tags:
- customer-support
- mistral
- qlora
- fine-tuned
- supportsphere
datasets:
- bitext/Bitext-customer-support-llm-chatbot-training-dataset
base_model: mistralai/Mistral-7B-v0.3
---

# SupportSphere — Mistral-7B Fine-tuned for Customer Support

Fine-tuned [Mistral-7B-v0.3](https://huggingface.co/mistralai/Mistral-7B-v0.3) on the
[Bitext customer support dataset](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset)
(26k examples) using QLoRA.

Part of the [SupportSphere](https://github.com/ayeshaimtiazzz/SupportSphere) production
multi-tenant AI customer support platform.

## Training Details
| Parameter | Value |
|-----------|-------|
| Base model | mistralai/Mistral-7B-v0.3 |
| Dataset | Bitext customer support (26k examples) |
| Method | QLoRA (4-bit NF4 + LoRA rank=16) |
| Hardware | NVIDIA T4 16GB (Google Colab free tier) |
| Precision | fp16 (T4 compatible) |
| Max steps | 500 |
| Effective batch | 16 (2 × grad_accum 8) |
| Learning rate | 2e-4 cosine |

## Intents Covered
order\_status · refund\_request · technical\_issue · billing · general\_faq · account · shipping · cancellation

## Usage
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "{HF_REPO_ID}"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

prompt = "<s>[INST] I want to return my order and get a refund. [/INST]"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=True)
print(tokenizer.decode(output[0], skip_special_tokens=True))
```
"""

HfApi().upload_file(
    path_or_fileobj=card.encode(),
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
)
print("Model card uploaded")
print(f"View your model: https://huggingface.co/{HF_REPO_ID}")


## Phase 3 Complete ✅

**What to record for your README A/B test table:**
- Final train loss (shown in Cell 12 output)
- Training time in minutes (shown in Cell 12 output)  
- GPU: NVIDIA T4 16GB
- Steps: 500

**Back in your local project**, add to `.env`:
```
HF_MODEL_ID=your_username/supportsphere-mistral-support
```

Then tell Claude and we'll wire the fine-tuned model into the Phase 4 agent.
